In [ ]:
# Unconditional diagnostic: runs on every push (free test push included),
# no KAGGLE_IS_COMPETITION_RERUN gate. Exercises the exact same dataset-mount
# path + checkpoint-loading logic that hypothesis_agent.py's _init_models
# uses, for the *object-identity* checkpoint's dataset version specifically,
# to catch a total setup failure before spending real submission quota.
import os
import shutil
import sys
import json
import traceback

print('=== step: listing /kaggle/input ===', flush=True)
for root, dirs, files in os.walk('/kaggle/input'):
    depth = root.count(os.sep)
    if depth > 4:
        continue
    print(root, '->', files[:10], flush=True)

SRC = '/kaggle/input/datasets/calamitychasm/jepa-hypothesis-agent'
print(f'=== step: checking {SRC} exists: {os.path.exists(SRC)} ===', flush=True)
if os.path.exists(SRC):
    print('contents:', os.listdir(SRC), flush=True)

try:
    print('=== step: copying jepa + checkpoints ===', flush=True)
    shutil.copytree(f'{SRC}/jepa', '/kaggle/working/jepa')
    shutil.copytree(f'{SRC}/checkpoints', '/kaggle/working/checkpoints')
    print('copy OK, checkpoints dir contents:', os.listdir('/kaggle/working/checkpoints'), flush=True)

    sys.path.insert(0, '/kaggle/working')
    import torch
    print(f'torch {torch.__version__}', flush=True)

    from jepa.device import get_device
    from jepa.models import CNNEncoder, MoEPredictor, ValueHead

    device = get_device()
    print(f'device: {device}', flush=True)

    print('=== step: loading encoder_moe.pt ===', flush=True)
    encoder = CNNEncoder().to(device)
    encoder.load_state_dict(torch.load('/kaggle/working/checkpoints/encoder_moe.pt', map_location=device))
    encoder.eval()
    print('encoder OK', flush=True)

    print('=== step: loading game_vocab_moe.json ===', flush=True)
    vocab_path = '/kaggle/working/checkpoints/game_vocab_moe.json'
    game_vocab = json.loads(open(vocab_path).read()) if os.path.exists(vocab_path) else {}
    num_games = max(len(game_vocab), 1)
    print(f'num_games={num_games}', flush=True)

    print('=== step: loading moe_predictor.pt ===', flush=True)
    predictor = MoEPredictor(num_games=num_games, num_experts=8).to(device)
    predictor.load_state_dict(torch.load('/kaggle/working/checkpoints/moe_predictor.pt', map_location=device))
    predictor.eval()
    print('predictor OK', flush=True)

    print('=== step: loading value_head.pt ===', flush=True)
    value_head = ValueHead().to(device)
    value_path = '/kaggle/working/checkpoints/value_head.pt'
    if os.path.exists(value_path):
        value_head.load_state_dict(torch.load(value_path, map_location=device))
        print('value_head OK', flush=True)
    else:
        print('WARNING: no value_head.pt found', flush=True)

    print('=== step: dummy forward pass through full pipeline ===', flush=True)
    import numpy as np
    from jepa.grid import arc3_frame_to_tensor, CANVAS
    dummy_grid = [[0] * 64 for _ in range(64)]
    tensor = arc3_frame_to_tensor([dummy_grid])
    x = torch.from_numpy(tensor).unsqueeze(0).to(device)
    with torch.no_grad():
        feat = encoder(x)
        action_t = torch.full((1,), 0, dtype=torch.long, device=device)
        xy_t = torch.zeros((1, 2), dtype=torch.float32, device=device)
        game_t = torch.full((1,), 0, dtype=torch.long, device=device)
        out = predictor(feat, action_t, xy_t, game_t)
        v = value_head(feat)
    print(f'forward pass OK, feat shape={feat.shape}, out type={type(out)}, v shape={v.shape}', flush=True)

    print('=== ALL DIAGNOSTICS PASSED ===', flush=True)
except Exception:
    print('=== DIAGNOSTIC FAILED, FULL TRACEBACK BELOW ===', flush=True)
    traceback.print_exc()

# Always write a dummy submission so this kernel is a valid non-rerun test push.
import pandas as pd
pd.DataFrame(
    data=[['1_0', '1', True, 1]],
    columns=['row_id', 'game_id', 'end_of_game', 'score'],
).to_parquet('/kaggle/working/submission.parquet', index=False)
